## Measure TPS of Gemma 4

In [6]:
import time
from ollama import chat
from typing import Optional

def measure_tokens_per_second(
    model: str = "gemma4",
    prompt: str = "Write a short story about a robot.",
    system_prompt: Optional[str] = None
) -> dict:
    """
    Measure tokens per second generation speed for an Ollama model.
    
    Args:
        model: Model name (default: gemma4)
        prompt: User prompt to send to the model
        system_prompt: Optional system prompt
    
    Returns:
        Dictionary with: tokens, seconds, tokens_per_second, full_response
    """
    
    messages = []
    
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    messages.append({"role": "user", "content": prompt})
    
    # Start timing
    start_time = time.time()
    token_count = 0
    full_response = ""
    
    # Stream the response and count tokens
    stream = chat(
        model=model,
        messages=messages,
        stream=True,
    )
    
    for chunk in stream:
        if chunk.message.content:
            full_response += chunk.message.content
            # Rough token count: split by whitespace (1 token ≈ 1 word)
            # This is approximate; for exact count, use model's token counter
            token_count += len(chunk.message.content.split())
    
    # Calculate elapsed time
    elapsed_time = time.time() - start_time
    tokens_per_second = token_count / elapsed_time if elapsed_time > 0 else 0
    
    results = {
        "tokens": token_count,
        "seconds": round(elapsed_time, 2),
        "tokens_per_second": round(tokens_per_second, 2),
        "full_response": full_response
    }
    
    return results

# Example usage
if __name__ == "__main__":
    results = measure_tokens_per_second(
        model="gemma4",
        prompt="Explain quantum computing in 100 words."
    )
    
    print(f"⏱️  Tokens Per Second Measurement")
    print(f"{'─' * 40}")
    print(f"Tokens generated: {results['tokens']}")
    print(f"Time elapsed: {results['seconds']}s")
    print(f"Throughput: {results['tokens_per_second']} tokens/second")
    print(f"{'─' * 40}")
    print(f"\nGenerated text:\n{results['full_response']}")

⏱️  Tokens Per Second Measurement
────────────────────────────────────────
Tokens generated: 116
Time elapsed: 35.36s
Throughput: 3.28 tokens/second
────────────────────────────────────────

Generated text:
Quantum computing utilizes quantum mechanics to process information, offering a massive leap beyond traditional classical computers. Instead of classical "bits" (which are 1 or 0), quantum computers use "qubits."

Qubits exploit two quantum phenomena: superposition, allowing them to exist in multiple states simultaneously, and entanglement, linking their fates together. This parallel processing power enables them to explore vast numbers of solutions at once.

While still in early stages, quantum computers promise to solve problems currently impossible, revolutionizing fields like drug discovery, materials science, and cryptography, by optimizing complex calculations with unprecedented speed.


In [7]:
# Quick test: Measure tokens per second for gemma4
results = measure_tokens_per_second(
    model="gemma4",
    prompt="Tell me a joke about programming.",
    system_prompt="You are a helpful AI assistant."
)

print(f"📊 Gemma4 Token Generation Speed")
print(f"{'=' * 50}")
print(f"✓ Tokens generated: {results['tokens']}")
print(f"✓ Time elapsed: {results['seconds']}s")
print(f"✓ Throughput: {results['tokens_per_second']} tokens/second")
print(f"{'=' * 50}")

📊 Gemma4 Token Generation Speed
✓ Tokens generated: 19
✓ Time elapsed: 1.66s
✓ Throughput: 11.45 tokens/second


In [ ]:
# Advanced: Multiple prompts benchmark
import statistics

def benchmark_multiple_prompts(model: str = "gemma4", num_tests: int = 3):
    """Run multiple prompts and calculate average TPS"""
    
    test_prompts = [
        "Explain machine learning in simple terms.",
        "What are the benefits of Python for data science?",
        "Describe the water cycle.",
    ]
    
    results_list = []
    
    for i, prompt in enumerate(test_prompts[:num_tests], 1):
        print(f"\n🔄 Test {i}/{num_tests}: {prompt[:40]}...")
        result = measure_tokens_per_second(model=model, prompt=prompt)
        results_list.append(result)
        print(f"   → {result['tokens_per_second']} tokens/sec")
    
    tps_values = [r['tokens_per_second'] for r in results_list]
    
    print(f"\n{'='*50}")
    print(f"📈 Benchmark Results for {model}")
    print(f"{'='*50}")
    print(f"Average TPS: {statistics.mean(tps_values):.2f}")
    print(f"Min TPS: {min(tps_values):.2f}")
    print(f"Max TPS: {max(tps_values):.2f}")
    print(f"Std Dev: {statistics.stdev(tps_values) if len(tps_values) > 1 else 0:.2f}")
    
    return results_list

# Uncomment to run benchmark
# benchmark_results = benchmark_multiple_prompts(model="gemma4", num_tests=3)

# Ollama – Local Model (gemma4)

Connect to a locally-running Ollama instance and send chat messages using the `ollama` Python package.

In [1]:
from ollama import chat

In [2]:
from ollama import chat

stream = chat(
    model="gemma4",
    messages=[{"role": "user", "content": "What is 17 × 23?"}],
    stream=True,
)

in_thinking = False
content = ""
thinking = ""
for chunk in stream:
    if chunk.message.thinking:
        if not in_thinking:
            in_thinking = True
            print("Thinking:\n", end="", flush=True)
        print(chunk.message.thinking, end="", flush=True)
        # accumulate the partial thinking
        thinking += chunk.message.thinking
    elif chunk.message.content:
        if in_thinking:
            in_thinking = False
            print("\n\nAnswer:\n", end="", flush=True)
        print(chunk.message.content, end="", flush=True)
        # accumulate the partial content
        content += chunk.message.content

    # append the accumulated fields to the messages for the next request
    new_messages = [{"role": "assistant", "thinking": thinking, "content": content}]

Thinking:
TheThe user has asked for the product of 17 and 23. I will perform the multiplication.

$17 \times 23$

Method 1: Standard multiplication
$17 \times 20 = 340$
$17 \times 3 = 51$
$340 + 51 = 391$

Method 2: Distributive property
$17 \times (20 + 3)$
$= 17 \times 20 + 17 \times 3$
$= 340 + 51$
$= 391$

The result is 391.

Answer:
$17 \times 23 = 391$

## Trying to summarize a websites

In [ ]:
from scraper import fetch_website_contents
from ollama import chat

url = "https://edwarddonner.com"
raw_text = fetch_website_contents(url)

# Keep prompt size manageable for local inference
max_chars = 12000
website_text = raw_text[:max_chars]

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful summarization assistant. "
            "Summarize clearly with sections: Overview, Key Points, and Takeaways."
        ),
    },
    {
        "role": "user",
        "content": (
            f"Summarize this website content from {url}. "
            "If some parts are noisy, focus on main ideas.\n\n"
            f"Website content:\n{website_text}"
        ),
    },
]

response = chat(model="gemma4", messages=messages)
summary = response["message"]["content"]

print("Website summary:\n")
print(summary)

Website summary:



NameError: name 'Markdown' is not defined